In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import yfinance as yf
%pip install neuralforecast
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.0/449.0 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 72.0 MB/s eta 0:00:00
  Attempting uninstall: tornado
    Found existing installation: tornado 6.5.1
    Uninstalling tornado-6.5.1:
      Successfully uninstalled tornado-6.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour i

In [ ]:
import pandas as pd

from google.colab import files
uploaded = files.upload()

Saving usd_idr_2020_2026.xlsx to usd_idr_2020_2026.xlsx


In [ ]:
data = pd.read_excel('Exchange Rate_2017_2026.xlsx',sheet_name='Sheet2', header = 0)
data = data[['ds', 'y', 'unique_id']]
data.head()

,ds,y,unique_id
0,2017-01-02,13512.0,IDR=X
1,2017-01-03,13506.0,IDR=X
2,2017-01-04,13443.0,IDR=X
3,2017-01-05,13348.0,IDR=X
4,2017-01-06,13319.0,IDR=X


In [ ]:
data['ds'] = pd.to_datetime(data['ds'])

train_df = data[data['ds'] < '2025-07-04']
test_df  = data[data['ds'] >= '2025-07-04']

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MAE

import numpy as np

def objective(trial):

    # =============================
    # Hyperparameter
    # =============================

    input_size = trial.suggest_categorical(
        "input_size", [42, 84]
    )

    learning_rate = trial.suggest_categorical(
        "learning_rate", [0.001, 0.005, 0.0001, 0.0005]
    )

    n_pool_kernel_size = trial.suggest_categorical(
        "n_pool_kernel_size",
        [[2, 2, 1], [4, 2, 1]]
    )

    max_steps = trial.suggest_categorical(
        "max_steps", [100, 150]
    )

    n_blocks = trial.suggest_categorical(
        "n_blocks",
        [[1, 1, 1], [2, 2, 2]]
    )

    mlp_units = trial.suggest_categorical(
        "mlp_units",
        [
            [[712, 712]] * 3,
            [[512, 512]] * 3,
            [[250, 250]] * 3
        ]
    )

    # =========================================
    # Time Series k-fold Cross Validation (k=5)
    # =========================================

    val_size = 151
    n_splits = 5

    initial_train_size = len(train_df) - (n_splits * val_size)

    mape_list = []

    for fold in range(n_splits):

        train_end = initial_train_size + fold * val_size

        train_fold = train_df.iloc[:train_end].copy()
        test_fold = train_df.iloc[
            train_end:train_end + val_size
        ].copy()

        horizon_fold = len(test_fold)

        # =============================
        # Model NHITS
        # =============================

        model = NHITS(
            h=horizon_fold,
            input_size=input_size,
            max_steps=max_steps,
            learning_rate=learning_rate,
            n_pool_kernel_size=n_pool_kernel_size,
            n_blocks=n_blocks,
            mlp_units=mlp_units,
            scaler_type="minmax",
        )

        nf = NeuralForecast(
            models=[model],
            freq="B"
        )

        # =============================
        # Training Model NHITS
        # =============================

        nf.fit(df=train_fold)

        # =============================
        # Peramalan Data Uji
        # =============================

        fcst = nf.predict()

        # =============================
        # Data Aktual dan Peramalan
        # =============================

        y_pred = fcst["NHITS"].values
        y_true = test_fold["y"].values

        # =============================
        # Hitung Nilai MAPE
        # =============================

        mape_fold = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        mape_list.append(mape_fold)

    # =============================
    # Rata-rata MAPE 5 Fold
    # =============================

    return np.mean(mape_list)

In [ ]:
import optuna

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    objective,
    n_trials=50)

[I 2026-06-06 02:43:37,894] A new study created in memory with name: no-name-5d8a6e1b-6b73-4af2-a856-48ffe487e539
/tmp/ipykernel_14262/3346501474.py:21: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [2, 2, 1] which is of type list.
  n_pool_kernel_size = trial.suggest_categorical(
/tmp/ipykernel_14262/3346501474.py:21: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [4, 2, 1] which is of type list.
  n_pool_kernel_size = trial.suggest_categorical(
/tmp/ipykernel_14262/3346501474.py:30: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [1, 1, 1] which is of type list.
  n_blocks = trial.suggest_categorical(
/tmp/ipykernel_14262/3346501474.py:30: UserWarning: Choices for a categorical distribution should be a tuple o

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=150` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


... (1,741 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

[I 2026-06-06 03:01:30,310] Trial 49 finished with value: 3.3148199220297463 and parameters: {'input_size': 84, 'learning_rate': 0.001, 'n_pool_kernel_size': [4, 2, 1], 'max_steps': 100, 'n_blocks': [2, 2, 2], 'mlp_units': [[712, 712], [712, 712], [712, 712]]}. Best is trial 24 with value: 2.2675539482161815.


In [ ]:
print("Best MAPE:", study.best_value)

print("Best Parameters:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

Best MAPE: 2.2675539482161815
Best Parameters:
input_size: 84
learning_rate: 0.0001
n_pool_kernel_size: [4, 2, 1]
max_steps: 100
n_blocks: [2, 2, 2]
mlp_units: [[712, 712], [712, 712], [712, 712]]
